# 04 — Results

The headline table, with the statistics done properly: paired over seeds,
bootstrap CIs, Wilcoxon signed-rank corrected by Holm-Bonferroni, and effect
sizes reported alongside p-values.

The baseline is the **tuned** sweep. Beating a deliberately weak incumbent
would prove nothing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from smartscan.config import load_config
from smartscan.eval.benchmark import leaderboard_markdown, run_benchmark

cfg = load_config("../configs/medium.yaml")
AGENTS = ["sequential", "random", "priority_rr", "ucb1", "thompson",
          "whittle", "coprime_sweep", "phase_locked"]
N_SEEDS = 12   # raise to 30 for the reported figures

result = run_benchmark(
    cfg.with_overrides(run={"n_seeds": N_SEEDS}),
    agents=AGENTS,
    metrics=["ttfi_hard_median_s", "twir_rate", "coverage", "staleness_max_s"],
    progress=False,
)
print(f"config hash {result.config_hash[:12]}, {N_SEEDS} paired seeds")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(leaderboard_markdown(result)))

## F1 — Waterfall with the scheduler's trajectory overlaid

The one figure that explains the whole project: ground truth in grey, the
receiver's tuned window as the bright band, confirmed intercepts as markers.

In [ ]:
from smartscan.agents import build_agent
from smartscan.env.rf_environment import build_episode, generate_scenario
from smartscan.runner import run_episode

seed = cfg.run.seed
scenario = generate_scenario(seed, config=cfg)
episode = build_episode(scenario)
show = ["sequential", "whittle", "coprime_sweep"]
runs = {k: run_episode(cfg, seed, build_agent(k, cfg, seed, scenario),
                       scenario=scenario, episode=episode) for k in show}

fig, axes = plt.subplots(len(show), 1, figsize=(14, 3.2 * len(show)), sharex=True)
time_s = np.arange(cfg.n_slots) * cfg.time.dt_s
truth = np.where(episode.occupancy > 0, 1.0, np.nan)

for ax, key in zip(axes, show):
    res = runs[key]
    ax.imshow(truth, aspect="auto", origin="lower", cmap="Greys", vmin=0, vmax=2,
              extent=[0, cfg.time.episode_s, 0, cfg.n_channels], interpolation="nearest")
    centre = res.actions.astype(float)
    ax.plot(res.dwell_slots * cfg.time.dt_s, centre, lw=0.4, color="tab:blue", alpha=0.8)
    ch, sl = np.nonzero(res.true_hit_mask)
    ax.scatter(sl * cfg.time.dt_s, ch, s=5, color="tab:red", zorder=3, label="intercept")
    ax.set_ylabel("channel")
    ax.set_title(f"{key} — {len(np.unique(episode.emitter_id[res.true_hit_mask])) - 0} emitters seen, "
                 f"{res.n_retunes} retunes", loc="left")
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

## F2 — TTFI survival curves (Kaplan-Meier)

Emitters never intercepted are **right-censored, not dropped**. Dropping them
is the standard error in this literature and flatters any scheduler that gives
up on hard targets.

In [ ]:
from smartscan.analysis.metrics import HARD_CLASSES, kaplan_meier, time_to_first_intercept

fig, ax = plt.subplots(figsize=(9, 4.6))
for key in show:
    durations, observed = [], []
    for s in range(cfg.run.seed, cfg.run.seed + 8):
        sc = generate_scenario(s, config=cfg)
        ep = build_episode(sc)
        r = run_episode(cfg, s, build_agent(key, cfg, s, sc), scenario=sc, episode=ep)
        d = time_to_first_intercept(ep, r.first_intercept, classes=HARD_CLASSES)
        durations.extend(d["durations_s"]); observed.extend(d["observed"])
    curve = kaplan_meier(np.asarray(durations), np.asarray(observed))
    ax.step(np.concatenate([[0], curve.times]),
            np.concatenate([[1.0], curve.survival]), where="post",
            label=f"{key} (median {curve.median:.2f} s, {curve.n_censored} censored)")
ax.set_xlabel("time since the emitter became interceptable (s)")
ax.set_ylabel("fraction not yet intercepted")
ax.set_title("Time to first intercept, scanning and agile emitters")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## F3 — Measured Pd against SNR, with exact binomial intervals

Clopper-Pearson rather than Wald: these estimates sit near 0 and 1, where a
normal approximation extends outside [0, 1] and under-covers.

In [ ]:
from smartscan.analysis.metrics import empirical_pd
from smartscan.env.propagation import p_detect

hard = load_config("../configs/hard.yaml")
sc = generate_scenario(hard.run.seed, config=hard)
ep = build_episode(sc)
res = run_episode(hard, hard.run.seed, build_agent("sequential", hard, 0, sc),
                  scenario=sc, episode=ep)
curve = empirical_pd(ep, res.visit_mask, res.true_hit_mask)
ok = curve["n"] > 30

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.errorbar(curve["snr_centre"][ok], curve["pd"][ok],
            yerr=[curve["pd"][ok] - curve["lo"][ok], curve["hi"][ok] - curve["pd"][ok]],
            fmt="o", capsize=3, label="measured (95% Clopper-Pearson)")
snr_axis = np.linspace(-15, 40, 300)
ax.plot(snr_axis, p_detect(snr_axis, n_integrate=1, pfa=hard.receiver.detector.pfa, swerling=1),
        "--", color="grey", label="analytic, 1 pulse Swerling I")
ax.set_xlabel("true SNR (dB)"); ax.set_ylabel("Pd"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Measured versus analytic detection performance")
plt.tight_layout()

## F5 — Interception ratio by emitter class

Where each scheduler's advantage actually comes from.

In [ ]:
from smartscan.analysis.metrics import average_intercept_rate

classes, table = set(), {}
for key in show:
    per = {}
    for s in range(cfg.run.seed, cfg.run.seed + 6):
        sc2 = generate_scenario(s, config=cfg)
        ep2 = build_episode(sc2)
        r = run_episode(cfg, s, build_agent(key, cfg, s, sc2), scenario=sc2, episode=ep2)
        for cls, rate in average_intercept_rate(ep2, r.true_hit_mask).items():
            if cls != "overall":
                per.setdefault(cls, []).append(rate)
                classes.add(cls)
    table[key] = {c: float(np.mean(v)) for c, v in per.items()}

classes = sorted(classes)
mat = np.array([[table[k].get(c, 0.0) for c in classes] for k in show])
fig, ax = plt.subplots(figsize=(1.3 * len(classes) + 3, 3.2))
im = ax.imshow(mat, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(classes)), classes, rotation=35, ha="right")
ax.set_yticks(range(len(show)), show)
ax.set_title("Intercepts per second, by emitter class")
fig.colorbar(im, ax=ax)
plt.tight_layout()

## Coverage versus exploitation

The trade-off a single metric hides. A scheduler can win on interception ratio
by parking and lose the band, which is why coverage entropy is reported beside
it.

In [ ]:
twir = result.per_agent("twir_rate")
cov = result.per_agent("coverage")
fig, ax = plt.subplots(figsize=(7, 5))
for key in AGENTS:
    if key in twir:
        ax.scatter(np.nanmedian(twir[key]), np.nanmedian(cov[key]), s=70)
        ax.annotate(key, (np.nanmedian(twir[key]), np.nanmedian(cov[key])),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("threat-weighted interception ratio  (exploitation)")
ax.set_ylabel("fraction of emitters ever found  (coverage)")
ax.set_title("Neither axis alone is the mission")
ax.grid(alpha=0.3)
plt.tight_layout()